In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import copy
import matplotlib.pyplot as plt

# --- 1. Adatok betöltése ---
data = np.load("20000.npy")
df = pd.DataFrame(data)
df["palya_id"] = df["eventID"].astype(str) + "_" + df["trackID"].astype(str)
primerek = df[df["parentID"] == 0]

parameterek = ['posX', 'posY', 'momDirX', 'momDirY', 'momDirZ', 'edep']
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 1000
BATCH_SIZE = 256

layers = [primerek[primerek["volumeID[3]"] == i].drop_duplicates(subset=["palya_id"]) for i in range(4)]

def prepare_layer_pair_data(layerA, layerB):
    common_labels = np.array(list(set(layerA["palya_id"]) & set(layerB["palya_id"])))
    np.random.shuffle(common_labels)
    split = int(0.9 * len(common_labels))
    train_labels, val_labels = common_labels[:split], common_labels[split:]

    l_train = [layerA[layerA["palya_id"].isin(train_labels)].set_index("palya_id").loc[train_labels],
               layerB[layerB["palya_id"].isin(train_labels)].set_index("palya_id").loc[train_labels]]
    l_val = [layerA[layerA["palya_id"].isin(val_labels)].set_index("palya_id").loc[val_labels],
             layerB[layerB["palya_id"].isin(val_labels)].set_index("palya_id").loc[val_labels]]

    x_mean = torch.tensor(l_train[0][parameterek].values.mean(axis=0), dtype=torch.float32)
    x_std = torch.tensor(l_train[0][parameterek].values.std(axis=0), dtype=torch.float32) + 1e-6
    return l_train, l_val, train_labels, val_labels, x_mean, x_std

# --- 3. Modell Definiálása ---
class MatchHead(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.ql = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU())
        self.kl = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU())
    def forward(self, x_prev, x_next):
        return self.ql(x_prev) @ self.kl(x_next).t()

class DualLayerMatcher(nn.Module):
    def __init__(self, input_dim, num_heads=3, hidden_dim=128):
        super().__init__()
        self.heads = nn.ModuleList([MatchHead(input_dim, hidden_dim) for _ in range(num_heads)])
        self.aggregator = nn.Linear(num_heads, 1)
    def match(self, x_prev, x_next, training=False):
        if training: x_prev = x_prev + 0.05 * torch.randn_like(x_prev) 
        dms = [head(x_prev, x_next) for head in self.heads] 
        final_dm = self.aggregator(torch.stack(dms, dim=-1)).squeeze(-1)
        return torch.softmax(final_dm, dim=1)

# --- 4. Általános Tanítási Függvény ---
def train_model_for_pair(pair_name, l_train, l_val, train_labels, val_labels, x_mean, x_std):
    print(f"\n{'='*40}\nTanítás indítása: {pair_name}\n{'='*40}")
    model = DualLayerMatcher(len(parameterek), num_heads=3, hidden_dim=128).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    best_acc, history = 0.0, []
    best_model_wts = copy.deepcopy(model.state_dict())

    for e in range(EPOCHS):
        model.train()
        np.random.shuffle(train_labels)
        for i in range(0, len(train_labels), BATCH_SIZE):
            batch_labels = train_labels[i:i+BATCH_SIZE]
            if len(batch_labels) < 2: continue
            xs = [(torch.tensor(df_lay.loc[batch_labels][parameterek].values, dtype=torch.float32) - x_mean) / x_std for df_lay in l_train]
            optimizer.zero_grad()
            probs = model.match(xs[0].to(DEVICE), xs[1].to(DEVICE), training=True)
            loss = nn.functional.nll_loss(torch.log(probs + 1e-8), torch.arange(len(batch_labels)).to(DEVICE))
            loss.backward(); optimizer.step()
        scheduler.step()

        if e % 100 == 0:
            model.eval()
            val_accs = []
            with torch.no_grad():
                for i in range(0, len(val_labels), 512):
                    v_batch = val_labels[i:i+512]
                    v_xs = [(torch.tensor(df_lay.loc[v_batch][parameterek].values, dtype=torch.float32) - x_mean) / x_std for df_lay in l_val]
                    acc = (torch.argmax(model.match(v_xs[0].to(DEVICE), v_xs[1].to(DEVICE)), dim=1) == torch.arange(len(v_batch)).to(DEVICE)).float().mean().item() * 100
                    val_accs.append(acc)
            mean_acc = np.mean(val_accs)
            history.append(mean_acc)
            if mean_acc > best_acc:
                best_acc, best_model_wts = mean_acc, copy.deepcopy(model.state_dict())
            print(f"Epoch {e:4d} | Val Acc: {mean_acc:.1f}% | Best: {best_acc:.1f}%")

    model.load_state_dict(best_model_wts)
    return model, history


l1_train, l1_val, t1, v1, m1, s1 = prepare_layer_pair_data(layers[0], layers[1])
model_L1_L2, hist1 = train_model_for_pair("L1-L2 Modell", l1_train, l1_val, t1, v1, m1, s1)



Tanítás indítása: L1-L2 Modell
Epoch    0 | Val Acc: 0.7% | Best: 0.7%
